# Stage 2 — Karst Susceptibility (Mole Creek prototype)

Builds the karst susceptibility layer: karst intensity (from `KCATEGORY`)
and exposure type (from `KEXPOSED`/`KCOVERED`/`KINTERSTR`), then
cross-checks against 1:25,000 geology as a plausibility check only — not
a scoring input (see README "Workflow / Methodology Reference": don't
double-count layers describing the same underlying thing).

**Field meanings, confirmed from the Karst Atlas v3 Data Dictionary**
(Sharples 2003), not assumed:

- `KCATEGORY`, `KEXPOSED`, `KCOVERED`, `KINTERSTR` all share the same
  ordinal scale: **A** = intensely karstified, **B** = substantially,
  **C** = partially, **D** = possibly partially, **0** = not karst.
- `KEXPOSED`/`KCOVERED`/`KINTERSTR` classify *which part* of a polygon's
  karst is exposed/covered/interstratal — confirmed **mutually
  exclusive** per polygon in the Mole Creek data (exactly one is
  non-zero, or all zero for non-karst polygons).
- Exposed/covered karst is autogenic (recharges from its own rainfall);
  interstratal karst is not (bounded by non-karst cover, so its
  hydrology comes from elsewhere) — this distinction matters for Stage 3
  (hydrological connectivity).

In [ ]:
from pathlib import Path
import os
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import rasterize
import matplotlib.pyplot as plt

data = Path(os.environ.get("AT3_DATA_DIR", "data"))
outpath = data / "Output_data"

karst_mc = gpd.read_file(str(outpath / "karst_mc_EPSG7855.gpkg"))
geology_mc = gpd.read_file(str(outpath / "geology_mc_EPSG7855.gpkg"))

# gpd.clip() in Stage 1 can leave degenerate slivers (line/point) where a
# polygon just grazes the Mole Creek boundary -- drop them, ~zero area, not meaningful
n_before = len(karst_mc)
karst_mc = karst_mc[karst_mc.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
print(f"Dropped {n_before - len(karst_mc)} degenerate clip artifacts (line/point geometries)")

## 2.1 Karst intensity — ordinal reclass of `KCATEGORY`

Treated as an ordinal scale, not a ratio — A isn't literally "4x more karstified" than D.

In [ ]:
CATEGORY_SCORE = {"A": 4, "B": 3, "C": 2, "D": 1, "0": 0}
karst_mc["karst_intensity"] = karst_mc["KCATEGORY"].map(CATEGORY_SCORE)
assert karst_mc["karst_intensity"].isna().sum() == 0, "unmapped KCATEGORY values found"

print(karst_mc["karst_intensity"].value_counts().sort_index())

## 2.2 Exposure type — categorical, from whichever of KEXPOSED/KCOVERED/KINTERSTR is non-zero

In [ ]:
def exposure_type(row):
    if row["KEXPOSED"] != "0":
        return "exposed"
    if row["KCOVERED"] != "0":
        return "covered"
    if row["KINTERSTR"] != "0":
        return "interstratal"
    return "non-karst"

karst_mc["exposure_type"] = karst_mc.apply(exposure_type, axis=1)
EXPOSURE_CODE = {"non-karst": 0, "interstratal": 1, "covered": 2, "exposed": 3}
karst_mc["exposure_code"] = karst_mc["exposure_type"].map(EXPOSURE_CODE)

print(karst_mc["exposure_type"].value_counts())

fig, ax = plt.subplots(figsize=(6, 6))
karst_mc.plot(ax=ax, column="exposure_type", legend=True, categorical=True, cmap="Set2")
plt.title("Mole Creek karst exposure type")
plt.show()

## 2.3 Geology cross-check (plausibility check only, not a score input)

Does mapped karst actually coincide with carbonate-described geology?
Broken down by exposure type deliberately — covered/interstratal karst
is *expected* to show low agreement, since by definition it isn't
visible at the surface (surface geology mapping shows the cover
sediment, not the karst bedrock underneath).

In [ ]:
overlay = gpd.overlay(karst_mc[karst_mc["karst_intensity"] > 0], geology_mc, how="intersection")
overlay["is_carbonate"] = overlay["description"].fillna("").str.lower().str.contains("limestone|dolomite|carbonate")
overlay["area"] = overlay.geometry.area

by_exposure = overlay.groupby("exposure_type").apply(
    lambda g: (g.loc[g["is_carbonate"], "area"].sum() / g["area"].sum()) * 100
)
print("% of overlap area matching carbonate-described surface geology, by exposure type:")
print(by_exposure.round(1))
print("\n(Low % for covered/interstratal is EXPECTED, not an error.)")

## 2.4 Rasterise onto the Stage 1 DEM template

First rasterisation step in the whole pipeline — geology and the boundary stay as vectors, only what's actually needed as a continuous surface gets rasterised.

In [ ]:
with rasterio.open(str(outpath / "dem_mc_EPSG7855.tif")) as dem_src:
    template_transform = dem_src.transform
    template_shape = (dem_src.height, dem_src.width)
    template_crs = dem_src.crs

intensity_raster = rasterize(
    [(geom, val) for geom, val in zip(karst_mc.geometry, karst_mc["karst_intensity"])],
    out_shape=template_shape, transform=template_transform, fill=0, dtype="uint8",
)
exposure_raster = rasterize(
    [(geom, val) for geom, val in zip(karst_mc.geometry, karst_mc["exposure_code"])],
    out_shape=template_shape, transform=template_transform, fill=0, dtype="uint8",
)

profile = {
    "driver": "GTiff", "height": template_shape[0], "width": template_shape[1],
    "count": 1, "dtype": "uint8", "crs": template_crs, "transform": template_transform, "nodata": 255,
}
with rasterio.open(outpath / "karst_intensity_mc_EPSG7855.tif", "w", **profile) as dst:
    dst.write(intensity_raster, 1)
with rasterio.open(outpath / "karst_exposure_mc_EPSG7855.tif", "w", **profile) as dst:
    dst.write(exposure_raster, 1)

karst_mc.to_file(outpath / "karst_mc_EPSG7855.gpkg", driver="GPKG")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(intensity_raster, cmap="viridis")
axes[0].set_title("Karst intensity (0-4)")
axes[1].imshow(exposure_raster, cmap="Set2")
axes[1].set_title("Exposure type (0-3)")
plt.show()

## Stage 2 outputs

- `karst_intensity_mc_EPSG7855.tif` — ordinal karst intensity raster (0-4)
- `karst_exposure_mc_EPSG7855.tif` — exposure type raster (0-3)
- `karst_mc_EPSG7855.gpkg` — updated with `karst_intensity`, `exposure_type`, `exposure_code` columns

**Next (Stage 3):** hydrological connectivity — `KPROXCATCH` as the
primary indicator (remembering it classifies the *downstream karst's
category*, not the amount of water/pressure — see data dictionary),
cross-checked against DEM-derived flow direction/accumulation.